# German Credit Scorecard Analysis (scorecardpl)

This notebook demonstrates end-to-end usage of `scorecardpl` on the classic German Credit dataset:
- Load and prepare the data
- Train/valid split + variable filtering
- WOE/IV binning (numeric + categorical)
- WOE transform and logistic regression scorecard
- Performance evaluation (AUC/KS), PSI, plots, and a simple report


In [2]:
%matplotlib inline
import os
import numpy as np
import pandas as pd
import polars as pl

from scorecardpl import (
    split_df, var_filter, woebin, woebin_ply,
    scorecard, scorecard_ply, perf_eva, iv_summary, psi,
    woebin_plot, bins_export_json, bins_import_json, write_method_report,
)


## Load German Credit data

We fetch the Statlog German Credit dataset from UCI. The label maps to 0/1 as: 1=good (0), 2=bad (1).

In [3]:
uci_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'
cols = [
    'Status', 'Duration', 'CreditHistory', 'Purpose', 'CreditAmount',
    'Savings', 'Employment', 'InstallmentRate', 'PersonalStatusSex', 'OtherDebtors',
    'ResidenceSince', 'Property', 'Age', 'OtherInstallmentPlans', 'Housing',
    'ExistingCredits', 'Job', 'Liables', 'Telephone', 'ForeignWorker', 'class'
]
df_pd = pd.read_csv(uci_url, sep=' ', header=None, names=cols)
# Map target: 1 (good) -> 0, 2 (bad) -> 1
df_pd['y'] = (df_pd['class'] == 2).astype(int)
df_pd = df_pd.drop(columns=['class'])
df_pl = pl.from_pandas(df_pd)
df_pl.head()


Status,Duration,CreditHistory,Purpose,CreditAmount,Savings,Employment,InstallmentRate,PersonalStatusSex,OtherDebtors,ResidenceSince,Property,Age,OtherInstallmentPlans,Housing,ExistingCredits,Job,Liables,Telephone,ForeignWorker,y
str,i64,str,str,i64,str,str,i64,str,str,i64,str,i64,str,str,i64,str,i64,str,str,i64
"""A11""",6,"""A34""","""A43""",1169,"""A65""","""A75""",4,"""A93""","""A101""",4,"""A121""",67,"""A143""","""A152""",2,"""A173""",1,"""A192""","""A201""",0
"""A12""",48,"""A32""","""A43""",5951,"""A61""","""A73""",2,"""A92""","""A101""",2,"""A121""",22,"""A143""","""A152""",1,"""A173""",1,"""A191""","""A201""",1
"""A14""",12,"""A34""","""A46""",2096,"""A61""","""A74""",2,"""A93""","""A101""",3,"""A121""",49,"""A143""","""A152""",1,"""A172""",2,"""A191""","""A201""",0
"""A11""",42,"""A32""","""A42""",7882,"""A61""","""A74""",2,"""A93""","""A103""",4,"""A122""",45,"""A143""","""A153""",1,"""A173""",2,"""A191""","""A201""",0
"""A11""",24,"""A33""","""A40""",4870,"""A61""","""A73""",3,"""A93""","""A101""",4,"""A124""",53,"""A143""","""A153""",2,"""A173""",2,"""A191""","""A201""",1


## Train/valid split and variable filtering

In [4]:
y = 'y'
xs = [c for c in df_pl.columns if c != y]
train, valid = split_df(df_pl, y=y, test_size=0.3, random_state=42)
train = var_filter(train, y=y, x=xs)
train.shape, valid.shape


((700, 21), (300, 21))

## WOE/IV binning

- Numeric: chi2 merging from fine quantiles, monotone WOE ('auto')
- Categorical: supervised merge to `cat_max_bins`

In [5]:
bins = woebin(
    train, y=y, x=[c for c in train.columns if c != y],
    bins=6, method='chi2', chi2_params={'init_bins': 60},
    monotonic='auto', cat_max_bins=5,
)
ivsum = iv_summary(bins)
ivsum


variable,iv,nbin
str,f64,i64
"""Status""",0.809315,4
"""CreditAmount""",0.58496,3
"""CreditHistory""",0.326533,5
"""Savings""",0.232083,5
"""Duration""",0.221583,3
…,…,…
"""ResidenceSince""",0.013592,3
"""Telephone""",0.012284,2
"""ExistingCredits""",0.00422,2


### Optional: save WOE plots per variable

In [ ]:
os.makedirs('plots/woe', exist_ok=True)
woe_paths = woebin_plot(bins, save_dir='plots/woe', show=False)
list(woe_paths.items())[:5]


## Transform to WOE features and train scorecard

In [6]:
train_w = woebin_ply(train, bins)
valid_w = woebin_ply(valid, bins)
sc = scorecard(bins, y=y, data=train_w)
X_valid = valid_w.select([c for c in valid_w.columns if c.endswith('_woe')]).to_numpy()
pred = sc.model.predict_proba(X_valid)[:, 1]
perf = perf_eva(valid_w[y], pred, plot='both', save_prefix='plots/german_credit')
perf


{'auc': 0.7500529100529102, 'ks': 0.43968253968253973}

### Apply scorecard points and inspect scores

In [7]:
scores = scorecard_ply(valid_w, sc.points_map)
scores.head(10)


score
f64
806.671901
783.740701
823.076376
745.967776
793.40305
795.827715
824.888579
788.214948
765.297321


## PSI between train and validation

In [8]:
psi_df = psi(train, valid, bins)
psi_df


variable,psi
str,f64
"""Purpose""",0.04939
"""CreditHistory""",0.041662
"""PersonalStatusSex""",0.028376
"""ResidenceSince""",0.020524
"""ForeignWorker""",0.019915
…,…
"""ExistingCredits""",0.002143
"""Housing""",0.001659
"""InstallmentRate""",0.001431


## Export bins and write a simple method report

In [9]:
os.makedirs('reports', exist_ok=True)
bins_export_json(bins, 'reports/german_credit_bins.json')
rows = [{
    'method': 'LR_WOE',
    'auc': float(perf['auc']),
    'ks': float(perf['ks']),
}]
write_method_report(rows, out_prefix='reports/german_credit_perf')


method,auc,ks
str,f64,f64
"""LR_WOE""",0.750053,0.439683


In [11]:
train_w

y,Status,Duration,CreditHistory,Purpose,CreditAmount,Savings,Employment,InstallmentRate,PersonalStatusSex,OtherDebtors,ResidenceSince,Property,Age,OtherInstallmentPlans,Housing,ExistingCredits,Job,Liables,Telephone,ForeignWorker,Status_bin,Status_woe,Duration_bin,Duration_woe,CreditHistory_bin,CreditHistory_woe,Purpose_bin,Purpose_woe,CreditAmount_bin,CreditAmount_woe,Savings_bin,Savings_woe,Employment_bin,Employment_woe,InstallmentRate_bin,InstallmentRate_woe,PersonalStatusSex_bin,PersonalStatusSex_woe,OtherDebtors_bin,OtherDebtors_woe,ResidenceSince_bin,ResidenceSince_woe,Property_bin,Property_woe,Age_bin,Age_woe,OtherInstallmentPlans_bin,OtherInstallmentPlans_woe,Housing_bin,Housing_woe,ExistingCredits_bin,ExistingCredits_woe,Job_bin,Job_woe,Liables_bin,Liables_woe,Telephone_bin,Telephone_woe,ForeignWorker_bin,ForeignWorker_woe
i64,str,i64,str,str,i64,str,str,i64,str,str,i64,str,i64,str,str,i64,str,i64,str,str,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64
0,"""A12""",24,"""A32""","""A42""",4351,"""A65""","""A73""",1,"""A92""","""A101""",4,"""A122""",48,"""A143""","""A152""",1,"""A172""",1,"""A192""","""A201""","""A12""",-0.380924,"""(11.0, inf]""",-0.172452,"""A32""",-0.066043,"""A42|A44|A45""",0.024541,"""(3868.0, inf]""",-0.587787,"""A65""",0.628609,"""A73""",-0.05884,"""(-inf, 2.0]""",0.246092,"""A92""",-0.175204,"""A101""",-0.006719,"""(3.0, inf]""",-0.037676,"""A122""",0.060259,"""(34.0, 63.0]""",0.28365,"""A143""",0.110171,"""A152""",0.17315,"""(-inf, 2.0]""",-0.011212,"""A172""",0.026231,"""(-inf, inf]""",0.0,"""A192""",0.141942,"""A201""",-0.044452
0,"""A11""",27,"""A34""","""A49""",2442,"""A61""","""A75""",4,"""A93""","""A101""",4,"""A123""",43,"""A142""","""A152""",4,"""A174""",2,"""A192""","""A201""","""A11""",-0.907316,"""(11.0, inf]""",-0.172452,"""A34""",0.712268,"""A46|A49""",-0.154151,"""(601.0, 3868.0]""",0.211784,"""A61""",-0.307227,"""A75""",0.204794,"""(3.0, inf]""",-0.189553,"""A93""",0.177455,"""A101""",-0.006719,"""(3.0, inf]""",-0.037676,"""A123""",-0.066553,"""(34.0, 63.0]""",0.28365,"""A142""",-0.336472,"""A152""",0.17315,"""(2.0, inf]""",0.376478,"""A174""",-0.154151,"""(-inf, inf]""",0.0,"""A192""",0.141942,"""A201""",-0.044452
0,"""A13""",21,"""A32""","""A40""",2923,"""A62""","""A73""",1,"""A92""","""A101""",1,"""A123""",28,"""A141""","""A152""",1,"""A174""",1,"""A192""","""A201""","""A13""",0.433636,"""(11.0, inf]""",-0.172452,"""A32""",-0.066043,"""A40""",-0.487924,"""(601.0, 3868.0]""",0.211784,"""A62""",0.108214,"""A73""",-0.05884,"""(-inf, 2.0]""",0.246092,"""A92""",-0.175204,"""A101""",-0.006719,"""(-inf, 2.0]""",-0.054305,"""A123""",-0.066553,"""(-inf, 34.0]""",-0.22712,"""A141""",-0.449996,"""A152""",0.17315,"""(-inf, 2.0]""",-0.011212,"""A174""",-0.154151,"""(-inf, inf]""",0.0,"""A192""",0.141942,"""A201""",-0.044452
0,"""A12""",9,"""A32""","""A43""",1206,"""A61""","""A75""",4,"""A92""","""A101""",4,"""A121""",25,"""A143""","""A152""",1,"""A173""",1,"""A191""","""A201""","""A12""",-0.380924,"""(-inf, 9.0]""",0.822765,"""A32""",-0.066043,"""A43""",0.388714,"""(601.0, 3868.0]""",0.211784,"""A61""",-0.307227,"""A75""",0.204794,"""(3.0, inf]""",-0.189553,"""A92""",-0.175204,"""A101""",-0.006719,"""(3.0, inf]""",-0.037676,"""A121""",0.449917,"""(-inf, 34.0]""",-0.22712,"""A143""",0.110171,"""A152""",0.17315,"""(-inf, 2.0]""",-0.011212,"""A173""",0.026231,"""(-inf, inf]""",0.0,"""A191""",-0.086629,"""A201""",-0.044452
0,"""A13""",6,"""A34""","""A40""",1323,"""A62""","""A75""",2,"""A91""","""A101""",4,"""A123""",28,"""A143""","""A152""",2,"""A173""",2,"""A192""","""A201""","""A13""",0.433636,"""(-inf, 9.0]""",0.822765,"""A34""",0.712268,"""A40""",-0.487924,"""(601.0, 3868.0]""",0.211784,"""A62""",0.108214,"""A75""",0.204794,"""(-inf, 2.0]""",0.246092,"""A91""",-0.545017,"""A101""",-0.006719,"""(3.0, inf]""",-0.037676,"""A123""",-0.066553,"""(-inf, 34.0]""",-0.22712,"""A143""",0.110

In [12]:
sc.points_map

{'Status': shape: (4, 4)
 ┌──────────┬─────┬───────────┬───────────┐
 │ variable ┆ bin ┆ woe       ┆ points    │
 │ ---      ┆ --- ┆ ---       ┆ ---       │
 │ str      ┆ str ┆ f64       ┆ f64       │
 ╞══════════╪═════╪═══════════╪═══════════╡
 │ Status   ┆ A11 ┆ -0.907316 ┆ -21.4577  │
 │ Status   ┆ A12 ┆ -0.380924 ┆ -9.008718 │
 │ Status   ┆ A14 ┆ 1.329886  ┆ 31.451334 │
 │ Status   ┆ A13 ┆ 0.433636  ┆ 10.255338 │
 └──────────┴─────┴───────────┴───────────┘,
 'Duration': shape: (3, 4)
 ┌──────────┬─────────────┬───────────┬───────────┐
 │ variable ┆ bin         ┆ woe       ┆ points    │
 │ ---      ┆ ---         ┆ ---       ┆ ---       │
 │ str      ┆ str         ┆ f64       ┆ f64       │
 ╞══════════╪═════════════╪═══════════╪═══════════╡
 │ Duration ┆ (-inf, 9.0] ┆ 0.822765  ┆ 13.061371 │
 │ Duration ┆ (9.0, 11.0] ┆ 2.410798  ┆ 38.271374 │
 │ Duration ┆ (11.0, inf] ┆ -0.172452 ┆ -2.737669 │
 └──────────┴─────────────┴───────────┴───────────┘,
 'CreditHistory': shape: (5, 4)
 ┌────